# Walmart — jak dobrze modele przewidziały sprzedaż?

Ocena na zbiorze testowym (ostatnie 8 tygodni, których model nie widział).
Predykcje są zapisywane automatycznie przez `run_experiments` do `results/predykcje/`,
więc ten notatnik niczego nie trenuje — tylko wczytuje i rysuje.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from src.config import setup_style
from src.data import load_raw, clean
from src.experiment import load_results, load_preds, PREDS_DIR
from src import plots

setup_style()
df_clean = clean(load_raw(), verbose=False)
print('Dostępne predykcje:', sorted(p.stem for p in PREDS_DIR.glob('*.csv')))

## 1. Wybór modeli do porównania

Nazwy jak w `results/eksperymenty.csv` (kolumny `dane`, `model`).

In [ ]:
DANE = 'bazowy'
WYBRANE = ['XGBoost (strojony)', 'LightGBM (strojony)', 'Sieć z embeddingami (strojony)']

preds = {m: load_preds(DANE, m) for m in WYBRANE}
plots.summary(preds).round(3)

`błąd sumy tyg. %` — o ile procent średnio myli się prognoza **łącznej** sprzedaży sieci w tygodniu
(błędy pojedynczych działów częściowo się znoszą, więc ta liczba jest zwykle dużo mniejsza niż MAE/średnia).

## 2. Łączna sprzedaż sieci: rzeczywista vs prognoza

In [ ]:
plots.plot_total(preds, history=df_clean)
plt.tight_layout(); plt.show()

## 3. Przykładowe działy w sklepach

In [ ]:
plots.plot_examples(preds, history=df_clean, n=6, seed=0)   # zmień seed -> inne przykłady
plt.show()

In [ ]:
# Konkretny sklep i dział
plots.plot_series(preds, store=1, dept=1, history=df_clean)
plt.tight_layout(); plt.show()

## 4. Przewidziana vs rzeczywista (każdy punkt = sklep × dział × tydzień)

In [ ]:
fig, axes = plt.subplots(1, len(preds), figsize=(6 * len(preds), 6), squeeze=False)
for ax, (name, p) in zip(axes[0], preds.items()):
    plots.plot_scatter(p, name, ax=ax)
plt.tight_layout(); plt.show()

## 5. Błędy: rozkład, w czasie, gdzie model myli się najbardziej

In [ ]:
best = plots.summary(preds).index[0]
plots.plot_errors(preds[best], best)
plt.show()

In [ ]:
plots.plot_error_by_week(preds)
plt.tight_layout(); plt.show()

In [ ]:
plots.plot_error_by(preds, by='Dept', top=20)
plt.tight_layout(); plt.show()

## 6. Wszystkie modele i warianty danych

In [ ]:
res = load_results()
res.sort_values('WMAE')[['dane', 'model', 'WMAE', 'MAE', 'RMSE', 'R2', 'czas_s']].head(20).reset_index(drop=True)